# LambdaMART: query groups, uncertainty and pairwise updates

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/lambdamart_ranking.ipynb)

Run the cells in order. The notebook reproduces the three-split table, paired-query bootstrap, ECDF, tree trajectory and pairwise-gradient animation. It downloads hash-checked data, runs on CPU with four threads and needs no credentials.

## Environment

Tested with Python 3.12, NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1, XGBoost 3.4.1 and Matplotlib 3.11.2. XGBoost 3.4.1 requires Python 3.12 or later. The assertion below reports an incompatible runtime before installation; on Colab select a compatible Python runtime if necessary.

In [ ]:
import sys
assert sys.version_info >= (3, 12), "This notebook's XGBoost pin needs Python >=3.12."
%pip -q install "numpy>=2,<3" "scipy>=1.14,<2" "scikit-learn>=1.6,<2" "xgboost==3.4.1" "matplotlib>=3.9,<4"

In [ ]:
import os
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[variable] = "4"
from pathlib import Path
from urllib.request import urlopen
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.datasets import load_svmlight_file
from sklearn.metrics import ndcg_score
from xgboost import XGBRanker, XGBRegressor
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

## Data and sparse-input semantics

The LightGBM sample has 3005 documents, 300 feature columns and 201 query groups. Source revision and SHA-256 checks make the download reproducible. The [source repository licence](https://github.com/lightgbm-org/LightGBM/blob/3ee03c26ba8c9e8ddc05fa8687717cae8a7d61d9/LICENSE) is MIT; files remain at their original host.

Keep the CSR representation used below. XGBoost treats absent sparse entries as **missing**, with a learned default branch. Densifying the matrix turns those entries into numerical zeros and changes the experiment.

In [ ]:
revision = "3ee03c26ba8c9e8ddc05fa8687717cae8a7d61d9"
base_url = f"https://raw.githubusercontent.com/lightgbm-org/LightGBM/{revision}/examples/lambdarank/"
expected_hashes = {
    "rank.train": "a0c7201c89120879c14a5059e091f441cbf2a29b8aaef363885ccb1a530448df",
    "rank.train.query": "40cb5fef637229311cd30fab13cb7fb633b023e86a87ff04ec55db420143d6af",
}
cache = Path("data/batch12-ranking")
cache.mkdir(parents=True, exist_ok=True)
for filename, expected in expected_hashes.items():
    path = cache / filename
    if not path.exists():
        path.write_bytes(urlopen(base_url + filename).read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
features, relevance = load_svmlight_file(str(cache / "rank.train"))
group_sizes = np.loadtxt(cache / "rank.train.query", dtype=int)
assert group_sizes.sum() == features.shape[0]
query_ids = np.repeat(np.arange(len(group_sizes)), group_sizes)
print("Rows, features, queries:", features.shape, len(group_sizes))

## Per-query metric and disjoint query splits

Use exponential gains, k=10, average prediction ties and score one for all-zero-relevance queries. Each split holds out 41 complete query groups. The metric returns queries in sorted ID order, preserving correspondence across methods for paired comparisons.

In [ ]:
def per_query_ndcg(labels, scores, queries):
    values = []
    for query in np.unique(queries):
        mask = queries == query
        gain = 2 ** labels[mask] - 1
        value = (1.0 if gain.max() == 0 else
                 ndcg_score([gain], [scores[mask]], k=10))
        values.append(value)
    return np.array(values)

methods = {"Regression": "reg:squarederror", "Pairwise": "rank:pairwise", "LambdaMART": "rank:ndcg"}
records = []
tree_trajectories = {}
for split_seed in [11, 22, 33]:
    shuffled_queries = np.random.default_rng(split_seed).permutation(np.unique(query_ids))
    test = np.isin(query_ids, shuffled_queries[:41])
    assert not (set(query_ids[test]) & set(query_ids[~test]))
    for method, objective in methods.items():
        estimator = XGBRegressor if method == "Regression" else XGBRanker
        model = estimator(objective=objective, n_estimators=100, max_depth=3,
                          learning_rate=0.08, tree_method="hist", n_jobs=4,
                          random_state=split_seed)
        fit_options = {} if method == "Regression" else {"qid": query_ids[~test]}
        model.fit(features[~test], relevance[~test], **fit_options)
        predictions = model.predict(features[test])
        query_scores = per_query_ndcg(relevance[test], predictions, query_ids[test])
        records.append({"seed": split_seed, "method": method, "scores": query_scores})
        if split_seed == 11:
            tree_trajectories[method] = [
                per_query_ndcg(relevance[test], model.predict(features[test], iteration_range=(0, count)), query_ids[test]).mean()
                for count in range(1, 101)
            ]
print("Seed   Regression   Pairwise   LambdaMART")
for seed in [11, 22, 33]:
    values = [next(row["scores"].mean() for row in records if row["seed"] == seed and row["method"] == method) for method in methods]
    print(seed, "  ".join(f"{value:.4f}" for value in values))
print("Mean", "  ".join(f"{np.mean([r['scores'].mean() for r in records if r['method']==method]):.4f}" for method in methods))

## Paired-query uncertainty

Subtract regression NDCG from LambdaMART NDCG for the **same query**. Resample whole paired query differences 10000 times. The percentile interval conditions on the fitted models and split; it does not include refitting variability or establish a population-wide confidence guarantee. Query independence can also fail if documents or users recur across groups. Differences within 1e-12 are treated as numerical ties.

In [ ]:
paired_results = []
print("Seed  Difference   SE      95% percentile interval       Better/Worse/Tied")
for seed in [11, 22, 33]:
    scores = {r["method"]: r["scores"] for r in records if r["seed"] == seed}
    difference = scores["LambdaMART"] - scores["Regression"]
    draws = np.random.default_rng(1000 + seed).integers(0, len(difference), (10000, len(difference)))
    interval = np.quantile(difference[draws].mean(axis=1), [0.025, 0.975])
    standard_error = difference.std(ddof=1) / np.sqrt(len(difference))
    counts = (sum(difference > 1e-12), sum(difference < -1e-12), sum(abs(difference) <= 1e-12))
    paired_results.append((seed, difference.mean(), standard_error, interval, counts))
    print(f"{seed}  {difference.mean():+.4f}  {standard_error:.4f}  [{interval[0]:+.4f}, {interval[1]:+.4f}]  {counts}")

## Objective comparison and query distributions

The first plot treats split seeds as categories. The ECDF is a marginal distribution; use the paired counts above to identify per-query wins and losses. The tree trajectory is descriptive test evaluation, not a criterion for choosing tree count.

In [ ]:
colours = ["#227C87", "#D88920", "#8C4F95"]
fig, ax = plt.subplots(figsize=(8, 4))
for index, (method, colour) in enumerate(zip(methods, colours)):
    values = [r["scores"].mean() for r in records if r["method"] == method]
    ax.scatter(np.arange(3) + (index - 1) * 0.16, values, color=colour, label=method)
ax.set(xticks=range(3), xticklabels=[11, 22, 33], xlabel="Query-split seed", ylabel="Mean query NDCG@10")
ax.legend(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for method, colour in zip(methods, colours):
    values = next(r["scores"] for r in records if r["seed"] == 11 and r["method"] == method)
    axes[0].step(np.sort(values), np.arange(1, len(values)+1)/len(values), where="post", label=method, color=colour)
    axes[1].plot(np.arange(1, 101), tree_trajectories[method], label=method, color=colour)
axes[0].set(xlabel="Query NDCG@10", ylabel="Cumulative query fraction", title="41 seed-11 test queries")
axes[1].set(xlabel="Trees", ylabel="Mean query NDCG@10", title="Held-out trajectory, fixed 100-tree comparison")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

## Toy pairwise loss and numerical gradient

This example adjusts four free document scores, not a feature-to-score model. A preferred pair contributes equal and opposite logistic loss derivatives. A central finite-difference check tests the displayed formula.

In [ ]:
labels = np.array([3, 0, 2, 1])
scores = np.array([-0.2, 0.4, 0.1, 0.3])
pairs = [(i, j) for i in range(4) for j in range(4) if labels[i] > labels[j]]
def pairwise_loss(values):
    return sum(np.logaddexp(0, -(values[i] - values[j])) for i, j in pairs)
def pairwise_gradient(values):
    gradient = np.zeros_like(values)
    for i, j in pairs:
        derivative = -expit(-(values[i] - values[j]))
        gradient[i] += derivative
        gradient[j] -= derivative
    return gradient

epsilon = 1e-6
basis = np.eye(4)
numerical = np.array([(pairwise_loss(scores + epsilon * direction) - pairwise_loss(scores - epsilon * direction)) / (2 * epsilon) for direction in basis])
gradient_error = np.max(abs(numerical - pairwise_gradient(scores)))
assert gradient_error < 1e-7
history = [scores.copy()]
for update in range(40):
    scores -= 0.15 * pairwise_gradient(scores)
    history.append(scores.copy())
history = np.array(history)
print(f"Maximum gradient discrepancy: {gradient_error:.3e}")
print(f"Toy loss: {pairwise_loss(history[0]):.4f} -> {pairwise_loss(history[-1]):.4f}")

## Pairwise updates animation

Forty updates appear with an explicit initial state and final state. The animation holds the final state. Correct ordering can be reached before logistic loss has stopped falling.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(range(4), history[0], color="#227C87")
ax.set(xticks=range(4), xticklabels=[f"Document {i+1}\nrelevance {label}" for i, label in enumerate(labels)], ylim=(history.min()-0.3, history.max()+0.3), ylabel="Score")
frames = list(range(0, 41, 2)) + [40] * 6
def show_scores(step):
    for bar, score in zip(bars, history[step]): bar.set_height(score)
    ax.set_title(f"Pairwise gradient updates: {step} of 40")
    return bars
animation = FuncAnimation(fig, show_scores, frames=frames, interval=250)
animation.save("lambdamart-pairs.gif", writer=PillowWriter(fps=4))
display(HTML(animation.to_jshtml())); plt.close(fig)

## Exercises and references

1. Change k to three and recompute paired differences.
2. Exclude all-zero queries and report the denominator.
3. Compare sparse inputs with dense inputs, explicitly recording missing-value semantics.
4. Use separate validation queries to tune tree count; leave test queries untouched.

[Burges (2010), RankNet to LambdaRank to LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf); [XGBoost ranking documentation](https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html).